# ESM2 as-standard model

Notebook draft to generate ESM2 embeddings from pylogeny-aware data. 

Tasks
- Import ESM checkpoint
- Import ESM tokenizer
- Import input data (target)
- Preprocess data
- Tokenize data
- Generate embeddings

Downstream tasks
- Run embeddings through classification head pre-trained using lower-level data for token classification
- Assess performance


In [26]:
# Dependancies and libraries
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
import torch.optim as optim

from transformers import AutoModel, AutoTokenizer

import esm

import pandas as pd

from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold

In [2]:
# ESM checkpoints
ESM = ['facebook/esm2_t48_15B_UR50D',
        'facebook/esm2_t36_3B_UR50D',
        'facebook/esm2_t33_650M_UR50D',
        'facebook/esm2_t30_150M_UR50D',
        'facebook/esm2_t12_35M_UR50D',
        'facebook/esm2_t6_8M_UR50D']

In [3]:
# Define checkpoint to be used
checkpoint = ESM[5]

In [4]:
# Create tokenizer and model objects
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModel.from_pretrained(checkpoint)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [5]:
# Import target data
df_target= pd.read_csv("/Users/harry/Documents/Data Science MSc/PROJECT/MScProject/Target_1769.csv")

# Remove extraneous columns
df_target = df_target.iloc[:,0:5]



In [6]:
# Aggreate rows and update format
df_target = df_target.groupby(['Info_protein_id', 'Info_group']).agg(
    sequence=('Info_AA', ''.join), 
    label=('Class', list), 
    position=('Info_pos', list))

In [7]:
df_target.shape
df_target['label'].str.len().agg(['mean','max'])

mean     410.571429
max     1871.000000
Name: label, dtype: float64

In [8]:
# Create a list of sequences
sequences = df_target['sequence'].tolist()

# Instantiate the tokenizer using the AA sequence lists as input to the tokenizer to create tokenized sequences
inputs = tokenizer(
    sequences,
    padding=True,
    truncation=True,
    max_length=1024,
    return_tensors="pt"
)

# Put the model into evaluation mode
model.eval()

# Generate embeddings for the 
with torch.inference_mode():
    outputs = model(**inputs)

In [9]:
# Verify shape of embedding
outputs.last_hidden_state.size()

torch.Size([21, 1024, 320])

In [10]:
# Freeze the weights in the model
for param in model.parameters():
    param.requires_grad = False

### Generate embeddings from ESM2 for lower level data to train classification head

In [42]:
# Import lower level data
df_lower= pd.read_csv("/Users/harry/Documents/Data Science MSc/PROJECT/MScProject/Lower_1763.csv")

# Remove extraneous columns
df_lower = df_lower.iloc[:,0:5]

# Add mask column where 1 assigned if labelled with pos/neg epitope and -100 if NaN
df_lower['mask'] = df_lower['Class'].isin([-1,1]).astype('int32')
df_lower['mask'] = df_lower['mask'].replace(0, -100)

# Change class column so nan = -100 and -1 = 0. Used later in loss function for classifier training
df_lower['Class'] = df_lower['Class'].fillna(-100).astype('int32')
df_lower['Class'] = df_lower['Class'].replace(-1, 0).astype('int32')

df_lower

,Info_protein_id,Info_pos,Info_AA,Info_group,Class,mask
0,P0A4V2.1,1,M,386.0,-100,-100
1,P0A4V2.1,2,Q,386.0,-100,-100
2,P0A4V2.1,3,L,386.0,-100,-100
3,P0A4V2.1,4,V,386.0,-100,-100
4,P0A4V2.1,5,D,386.0,-100,-100
...,...,...,...,...,...,...
139122,YP_002644961.1,321,S,408.0,0,1
139123,YP_002644961.1,322,L,408.0,0,1
139124,YP_002644961.1,323,G,408.0,0,1
139125,YP_002644961.1,324,A,408.0,0,1


In [43]:
# Aggregate columns for wide format
df_lower = df_lower.groupby(['Info_protein_id','Info_group'], as_index=False).agg(
    sequence=('Info_AA', ''.join), 
    label=('Class', list), 
    position=('Info_pos', list),
    mask=('mask', list))

df_lower

,Info_protein_id,Info_group,sequence,label,position,mask
0,A1KFU9.1,544.0,MAENSNIDDIKAPLLAALGAADLALATVNELITNLRERAEETRTDT...,"[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, -100, -100, -100, -100, -10..."
1,A43589,594.0,MLGNAPSVVPNTTLGMHCGSFGSAPSNGWLKLGLVEFGGVAKLNAE...,"[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, -100, -100, -100, -100, -10..."
2,AAA21416.1,46.0,MLEGCILADSRQSKTAASPSPSRPQSSSNNSVPGAPNRVSFAKLRE...,"[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, -100, -100, -100, -100, -10..."
3,AAA21417.1,578.0,MLDVNFFDELRIGLATAEDIRQWSYGEVKKPETINYRTLKPEKDGL...,"[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, -100, -100, -100, -100, -10..."
4,AAA25359.1,410.0,MTDVSRKIRAWGRRLMIGTAAAVVLPGLVGLAGGAATAGAFSRPGL...,"[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, -100, -100, -100, -100, -10..."
...,...,...,...,...,...,...
334,YP_178023.1,632.0,MTEQQWNFAGIEAAASAIQGNVTSIHSLLDEGKQSLTKLAAAWGGS...,"[-100, -100, -100, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
335,YP_976577.1,178.0,MAKTIAYDEEARRGLERGLNALADAVKVTLGPKGRNVVLEKKWGAP...,"[-100, -100, -100, -100, -100, -100, 1, 1, 1, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, -100, -100, -100, 1, 1, 1, ..."
336,ZP_03425930.1,172.0,MAEELHAAAGSFASVTTGLAGDAWHGPASLAMTRAASPYVGWLNTA...,"[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, -100, -100, -100, -100, -10..."
337,ZP_03432777.1,91.0,MTDRVSVGNLRIARVLYDFVNNEALPGTDIDPDSFWAGVDKVVADL...,"[-100, -100, -100, -100, -100, -100, -100, -10...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[-100, -100, -100, -100, -100, -100, -100, -10..."


In [44]:
# Create a class column that checks whether the sequence contains a positive or negative epitope and apply a class column for the stratified grouped k fold
df_lower["class"] = df_lower["label"].apply(lambda x: 1 if 1 in x else -1)

# Check max length of sequences
df_lower['label'].str.len().agg(['mean','max'])

mean     402.427729
max     3186.000000
Name: label, dtype: float64

Df contains sequences with length > ESM max input length (1024), sliding window will need to be applied once draft complete - same applies for target data

In [45]:
# Info_group as the grouping variable and Class  / label as the stratification variable.
X = df_lower.index
y = df_lower['class']
groups = df_lower['Info_group']

# Instantiate GroupShuffleSplit instance to create grouped train/test splits, use 20% of the data for a hold out/test set
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)

# Split the data into train/test splits, create indices to be used to assign train/test labels to the df
train_cv_idx, test_idx = next(gss.split(X, y, groups))

# Create a train/test column in the dataframe and set the values of the test rows to train or test
df_lower.loc[test_idx, 'train_test'] = 'test'
df_lower.loc[train_cv_idx, 'train_test'] = 'train'

# Update X, y and groups with the remaining train_cv_idx indices to use in Statified Grouped k fold
X_train, y_train, groups_train = X[train_cv_idx], y[train_cv_idx], groups[train_cv_idx]


In [46]:
# Split into different folds ensuring stratification accross groups
sgkf = StratifiedGroupKFold(n_splits=5)

X_train_df = pd.DataFrame(index=range(0,len(df_lower)))
                                     
for fold, (train_idx, val_idx) in enumerate(sgkf.split(X, y, groups)):
    X_train_df.loc[train_idx, f'training_split {fold+1}'] = 1
    X_train_df.loc[val_idx, f'training_split {fold+1}']= 2

df_lower = df_lower.merge(X_train_df, left_index=True, right_index=True)

In [47]:
# Create train/val/test splits 
df_lower_cv = df_lower[df_lower['train_test'] == 'train'].reset_index()
df_lower_test = df_lower[df_lower['train_test'] == 'test'].reset_index()

df_lower_cv.shape

(261, 14)

In [48]:
# Create custom dataset class
class SequenceDataset(torch.utils.data.Dataset):
    
    def __init__(self, df):
        self.protein_id = df['Info_protein_id']
        self.sequence = df['sequence']
        self.position = df['position']
        self.labels = df['label']
        self.label_mask = df['mask']
        
    def __len__(self):
        return len(self.sequence)
        
    def __getitem__(self, idx):
        return {
            'protein_id': self.protein_id[idx],
            'sequence': self.sequence[idx],
            'position': self.position[idx],
            'label': self.labels[idx],
            'mask': self.label_mask[idx]
        }

lower_dataset = SequenceDataset(df_lower_cv)

Generate embeddings for each of the lower level sequences 

In [49]:
# Create a custom collator to maintain length of items within the batch
def collate_fn(batch):
    return {
        'protein_id': [x['protein_id'] for x in batch],
        'sequence': [x['sequence'] for x in batch],
        'position': [x['position'] for x in batch],
        'label': [x['label'] for x in batch],
        'mask': [x['mask'] for x in batch],
    }

# Create a DataLoader instance using the cv dataset and the custom collate function
loader = DataLoader(
    lower_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn
)

emb_output_list = []

# Loop through each batch of tensors and apply tokenisation to each sequence, using max length of 1024 (will require windowing and averaging in later versions)
for batch in loader:

    inputs = tokenizer(
        batch['sequence'],
        padding=True,
        truncation=True,
        max_length=1024,
        return_tensors='pt'
    )

    # Freeze model weights and calculate embeddings for the tokenized embeddings (remove start and end CLS/EOS tokens)
    with torch.no_grad():
        embeddings = model(**inputs).last_hidden_state[:,1:-1,:]

    # Convert each label and mask to tensors 
    label = [torch.tensor(item) for item in batch['label']]
    mask = [torch.tensor(item) for item in batch['mask']]
    
    # Pad each label and mask sequence to the size of the largest embedding within the batch
    label = pad_sequence(label, padding_value=-100, padding_side='right')[0:embeddings.size()[1]]
    mask = pad_sequence(mask, padding_value=-100, padding_side='right')[0:embeddings.size()[1]]

    print('embeddings', embeddings.size())
    print('label', label.size())
    print('mask', mask.size())
    
    # Append the embeddings, label and masks per batch to an output list 
    emb_output_list.append({'embeddings': embeddings,
                     'label': label, 
                     'mask': mask})

embeddings torch.Size([16, 729, 320])
label torch.Size([729, 16])
mask torch.Size([729, 16])
embeddings torch.Size([16, 1022, 320])
label torch.Size([1022, 16])
mask torch.Size([1022, 16])
embeddings torch.Size([16, 741, 320])
label torch.Size([741, 16])
mask torch.Size([741, 16])
embeddings torch.Size([16, 709, 320])
label torch.Size([709, 16])
mask torch.Size([709, 16])
embeddings torch.Size([16, 1022, 320])
label torch.Size([1022, 16])
mask torch.Size([1022, 16])
embeddings torch.Size([16, 671, 320])
label torch.Size([671, 16])
mask torch.Size([671, 16])
embeddings torch.Size([16, 1022, 320])
label torch.Size([1022, 16])
mask torch.Size([1022, 16])
embeddings torch.Size([16, 784, 320])
label torch.Size([784, 16])
mask torch.Size([784, 16])
embeddings torch.Size([16, 1022, 320])
label torch.Size([1022, 16])
mask torch.Size([1022, 16])
embeddings torch.Size([16, 801, 320])
label torch.Size([801, 16])
mask torch.Size([801, 16])
embeddings torch.Size([16, 664, 320])
label torch.Size([66

### Train classification head on lower level data

Tasks
- Instantiate classifier for token classification
- Train
- Test

In [50]:
# Create simple 2 layer neural network in PyTorch to return logits per residue representing each class
class PerResidueClassifier(nn.Module):
    def __init__(self, in_features=320, hidden_size=128, out_features=2):
        super().__init__()
        
        self.linear1 = nn.Linear(in_features, hidden_size)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(hidden_size, out_features)
        
    def forward(self, embeddings):
        x = self.linear1(embeddings)
        x = self.relu(x)
        logits = self.linear2(x)
        
        return logits

In [122]:
# Instantiate classifier
clf = PerResidueClassifier()

# # Create classifer out
# clf_outputs = []

# for emb_batch in emb_output_list:
#     outputs = clf(emb_batch['embeddings'])
#     clf_outputs.append({'clf_logits': outputs,
#                         'label': emb_batch['label'],
#                         'mask': emb_batch['mask']}
#                       )    

tensor(0.6945, grad_fn=<NllLossBackward0>)
tensor(0.6997, grad_fn=<NllLossBackward0>)
tensor(0.6941, grad_fn=<NllLossBackward0>)
tensor(0.6956, grad_fn=<NllLossBackward0>)
tensor(0.7059, grad_fn=<NllLossBackward0>)
tensor(0.6878, grad_fn=<NllLossBackward0>)
tensor(0.7004, grad_fn=<NllLossBackward0>)
tensor(0.6922, grad_fn=<NllLossBackward0>)
tensor(0.6963, grad_fn=<NllLossBackward0>)
tensor(0.6921, grad_fn=<NllLossBackward0>)
tensor(0.6959, grad_fn=<NllLossBackward0>)
tensor(0.6989, grad_fn=<NllLossBackward0>)
tensor(0.6983, grad_fn=<NllLossBackward0>)
tensor(0.6983, grad_fn=<NllLossBackward0>)
tensor(0.7007, grad_fn=<NllLossBackward0>)
tensor(0.6879, grad_fn=<NllLossBackward0>)
tensor(0.6536, grad_fn=<NllLossBackward0>)


In [146]:
# Instantiate cross-entropy loss function, ignores positions with mask -100
loss_fcn = nn.CrossEntropyLoss(ignore_index=-100)

# Adam optimiser
optimiser = optim.Adam(clf.parameters(), lr=0.001)

epochs = 20

# Training loop
for epoch in range(0, epochs):
    for i, batch in enumerate(emb_output_list):
        inputs = batch['embeddings']
        labels = batch['label']
    
        # Zero model gradients per every batch
        optimiser.zero_grad()
    
        # Caluclate logits by passing embeddings through the classifier
        outputs = clf(inputs)
    
        # Compute loss and gradients        
        loss = loss_fcn(outputs.reshape(-1, 2), labels.reshape(-1))

        # Calculate the gradients through the network
        loss.backward()

        # Adjust learning weights
        optimiser.step()

In [ ]:
# Make predictions using validation data